Install dependencies

In [ ]:
#  Install everything needed
!pip install -q \
    transformers==4.40.0 \
    soundfile==0.12.1 \
    datasets==2.19.0 \
    evaluate==0.4.1 \
    jiwer==3.0.3 \
    accelerate==0.29.3 \
    soundfile==0.12.1 \
    librosa==0.10.1 \
    audiomentations==0.33.0 \
    torch-audiomentations==0.11.0

# Verify key imports
import transformers, datasets, evaluate
print(f"transformers : {transformers.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"evaluate     : {evaluate.__version__}")

In [ ]:
# Fix fsspec to satisfy both gcsfs and datasets
!pip install -q "fsspec==2024.6.1" --upgrade

# Pin transformers to a version that satisfies sentence-transformers
# without breaking our other dependencies
!pip install -q "transformers==4.41.2" --upgrade

# Re-verify
import importlib
for pkg in ["transformers", "datasets", "evaluate", "fsspec"]:
    mod = importlib.import_module(pkg)
    print(f"{pkg:20s} : {mod.__version__}")
print("Conflicts resolved")

Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Checkn Drive

import os

DRIVE_ROOT = "/content/drive/MyDrive"

print("Top-level folders in MyDrive:")
for item in sorted(os.listdir(DRIVE_ROOT)):
    full = os.path.join(DRIVE_ROOT, item)
    kind = "DIR " if os.path.isdir(full) else "FILE"
    print(f"  [{kind}] {item}")

In [ ]:
# check the SALT_KD folder
import os

def tree(path, indent=0, max_depth=4):
    if indent > max_depth:
        return
    try:
        items = sorted(os.listdir(path))
    except PermissionError:
        return
    for item in items:
        full = os.path.join(path, item)
        kind = "DIR " if os.path.isdir(full) else "FILE"
        print("  " * indent + f"[{kind}] {item}")
        if os.path.isdir(full):
            tree(full, indent + 1, max_depth)

tree("/content/drive/MyDrive/SALT_KD")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT = "/content/drive/MyDrive/SALT_KD"

# Teacher models
TEACHER_W2V_PATH = None  # not found — needs uploading or downloading
TEACHER_WSP_PATH = f"{DRIVE_ROOT}/cache/transformers/models--Sunbird--asr-whisper-large-v2-salt/snapshots/54f8e3f92895e0ac66f1baa1262eca85b5eccf74"

# SALT data — load via HuggingFace datasets cache
HF_CACHE_DIR     = f"{DRIVE_ROOT}/cache"

# Student outputs
OUTPUT_W2V       = f"{DRIVE_ROOT}/outputs/wav2vec2_student"
OUTPUT_WHISPER   = f"{DRIVE_ROOT}/outputs/whisper_student"

os.makedirs(OUTPUT_W2V,    exist_ok=True)
os.makedirs(OUTPUT_WHISPER, exist_ok=True)

print("Paths resolved:")
for name, path in [
    ("Teacher Whisper", TEACHER_WSP_PATH),
    ("HF Cache (SALT)", HF_CACHE_DIR),
    ("Output W2V",      OUTPUT_W2V),
    ("Output Whisper",  OUTPUT_WHISPER),
]:
    status = "FOUND" if path and os.path.exists(path) else "NOT FOUND"
    print(f"  {name}: {status}  →  {path}")

In [ ]:
import os

os.environ["HF_DATASETS_CACHE"] = HF_CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE_DIR

checks = {
    "Teacher Whisper snapshot": TEACHER_WSP_PATH,
    "HF Cache dir":             HF_CACHE_DIR,
    "Output Wav2Vec2":          OUTPUT_W2V,
    "Output Whisper":           OUTPUT_WHISPER,
    "SALT lug dataset":         f"{HF_CACHE_DIR}/datasets/Sunbird___salt/multispeaker-lug/0.0.0",
}

all_ok = True
for name, path in checks.items():
    ok = path and os.path.exists(path)
    print(f"  {'Okay' if ok else 'Not Okay'} {name}: {path}")
    if not ok:
        all_ok = False

print("\n All paths OK — ready to proceed." if all_ok else "\n  Fix missing paths before continuing.")

In [ ]:
import os

DRIVE_ROOT = "/content/drive/MyDrive/SALT_KD"

def show_tree(root, indent=0):
    try:
        items = sorted(os.listdir(root))
        if not items:
            print(f"{'  ' * indent}  (empty)")
        for item in items:
            full = os.path.join(root, item)
            icon = "📁" if os.path.isdir(full) else "📄"
            size = ""
            if os.path.isfile(full):
                mb = os.path.getsize(full) / 1e6
                size = f"  ({mb:.1f} MB)"
            print(f"{'  ' * indent}{icon} {item}{size}")
            if os.path.isdir(full) and indent < 3:
                show_tree(full, indent + 1)
    except PermissionError:
        print(f"{'  ' * indent}  [permission denied]")

print(f"📁 SALT_KD/")
show_tree(DRIVE_ROOT)

Set correct paths and cache

In [ ]:
import os, torch

DRIVE_ROOT     = "/content/drive/MyDrive/SALT_KD"
CACHE_DIR      = f"{DRIVE_ROOT}/cache"
OUTPUT_W2V     = f"{DRIVE_ROOT}/outputs/wav2vec2_student"
OUTPUT_WHISPER = f"{DRIVE_ROOT}/outputs/whisper_student"

for path in [CACHE_DIR, OUTPUT_W2V, OUTPUT_WHISPER]:
    os.makedirs(path, exist_ok=True)

# Redirect HF cache to Drive — models persist across Colab sessions
os.environ["HF_HOME"]            = CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = f"{CACHE_DIR}/transformers"
os.environ["HF_DATASETS_CACHE"]  = f"{CACHE_DIR}/datasets"

DEVICE = torch.device("cuda")

print("  Paths configured")
print(f"   Cache   → {CACHE_DIR}  (persists across sessions)")
print(f"   W2V out → {OUTPUT_W2V}")
print(f"   WSP out → {OUTPUT_WHISPER}")

Load dataset

In [ ]:
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset, Audio, concatenate_datasets


#  Authenticate with your HF token from Colab secrets
HF_TOKEN = userdata.get("appauli_sunbird_access_token")
login(token=HF_TOKEN, add_to_git_credential=False)
print("Logged in to Hugging Face")


LANGUAGES = ['lug', 'ach', 'nyn', 'teo', 'lgg', 'eng']   # multispeaker ASR
SWA_LANG  = 'swa'                                           # studio only

print("Loading multispeaker splits (6 languages)...")
salt_splits = {}

for lang in LANGUAGES:
    ds = load_dataset(
        "Sunbird/salt",
        f"multispeaker-{lang}",
        cache_dir=f"{CACHE_DIR}/datasets",
        trust_remote_code=True,
        token=HF_TOKEN,
    )
    ds = ds.cast_column("audio", Audio(sampling_rate=16_000))
    salt_splits[lang] = ds
    print(f"  {lang}: train={len(ds['train'])}, dev={len(ds['dev'])}, test={len(ds['test'])}")

# Load Swahili from studio split
print("\nLoading studio-swa (Swahili)...")
try:
    swa_ds = load_dataset(
        "Sunbird/salt",
        "studio-swa",
        cache_dir=f"{CACHE_DIR}/datasets",
        trust_remote_code=True,
        token=HF_TOKEN,
    )
    swa_ds = swa_ds.cast_column("audio", Audio(sampling_rate=16_000))
    salt_splits["swa"] = swa_ds
    LANGUAGES_ALL = LANGUAGES + ["swa"]

    for split in swa_ds.keys():
        print(f"   swa ({split}): {len(swa_ds[split])} samples")

    # Check swa column names (may differ from multispeaker)
    swa_sample = swa_ds[list(swa_ds.keys())[0]][0]
    print(f"\n  swa columns : {list(swa_sample.keys())}")

except Exception as e:
    print(f"   studio-swa failed: {e}")
    print("   Proceeding with 6 languages only")
    LANGUAGES_ALL = LANGUAGES

#  Update label map to include all loaded languages
ID2LABEL = {i: lang for i, lang in enumerate(sorted(LANGUAGES_ALL))}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
NUM_LABELS = len(ID2LABEL)

print(f"\n Final language set ({NUM_LABELS} languages) ")
for i, lang in ID2LABEL.items():
    split_info = ""
    if lang in salt_splits:
        train_n = len(salt_splits[lang].get("train", []))
        split_info = f"  train={train_n}"
    print(f"  {i}: {lang}{split_info}")

#  SALT language tokens for Whisper
SALT_LANG_TOKENS = {
    'eng': 50259,
    'ach': 50357,
    'lgg': 50356,
    'lug': 50355,
    'nyn': 50354,
    'teo': 50353,
    'swa': 50350,
}

print("\n Dataset loading complete")

load whisper teacher

In [ ]:
import torch
import torch.nn as nn
from transformers import WhisperForConditionalGeneration, WhisperProcessor

WHISPER_ID = "Sunbird/asr-whisper-large-v2-salt"
DEVICE     = torch.device("cuda")

print(f"Loading Whisper teacher: {WHISPER_ID}")
print("~3GB download — saved to Drive cache after first run\n")

processor = WhisperProcessor.from_pretrained(
    WHISPER_ID,
    cache_dir=f"{CACHE_DIR}/transformers",
    token=HF_TOKEN,
)

teacher_whisper = WhisperForConditionalGeneration.from_pretrained(
    WHISPER_ID,
    torch_dtype=torch.float16,
    cache_dir=f"{CACHE_DIR}/transformers",
    token=HF_TOKEN,
).to(DEVICE)

teacher_whisper.eval()
for p in teacher_whisper.parameters():
    p.requires_grad = False

vram   = torch.cuda.memory_allocated() / 1e9
params = sum(p.numel() for p in teacher_whisper.parameters()) / 1e6
print(f" Whisper teacher loaded")
print(f"   Params    : {params:.0f}M")
print(f"   VRAM used : {vram:.2f} GB")
print(f"   Vocab size: {teacher_whisper.config.vocab_size}")

Build Whisper student + init from teacher

In [ ]:
from transformers import WhisperConfig, WhisperForConditionalGeneration

# Match teacher d_model so weights actually transfer
# Compress via fewer layers only (4 enc + 4 dec instead of 32+32)
student_config = WhisperConfig(
    d_model=1280,                   # must match teacher for weight transfer
    encoder_layers=4,
    decoder_layers=4,
    encoder_attention_heads=20,     # must match teacher (1280/64=20)
    decoder_attention_heads=20,
    encoder_ffn_dim=5120,           # must match teacher (4×d_model)
    decoder_ffn_dim=5120,
    vocab_size=teacher_whisper.config.vocab_size,
    max_source_positions=1500,
    max_target_positions=448,
    pad_token_id=teacher_whisper.config.pad_token_id,
    bos_token_id=teacher_whisper.config.bos_token_id,
    eos_token_id=teacher_whisper.config.eos_token_id,
    decoder_start_token_id=teacher_whisper.config.decoder_start_token_id,
)
student_whisper = WhisperForConditionalGeneration(student_config).to(DEVICE)

def init_student_from_teacher(teacher, student):
    print("Initializing student from teacher weights (layer dropping)...")

    # Copy non-layer weights (embeddings, layer norms, proj_out)
    shared_parts = [
        ("embed positions (enc)",
         teacher.model.encoder.embed_positions,
         student.model.encoder.embed_positions),
        ("embed positions (dec)",
         teacher.model.decoder.embed_positions,
         student.model.decoder.embed_positions),
        ("proj out",
         teacher.proj_out,
         student.proj_out),
    ]
    for name, t_mod, s_mod in shared_parts:
        try:
            s_mod.load_state_dict(t_mod.state_dict(), strict=True)
            print(f"  {name}: copied")
        except Exception as e:
            print(f"  {name}: skipped ({e})")

    # Layer dropping
    for part, t_layers, s_layers in [
        ("encoder", teacher.model.encoder.layers, student.model.encoder.layers),
        ("decoder", teacher.model.decoder.layers, student.model.decoder.layers),
    ]:
        n_t, n_s = len(t_layers), len(s_layers)
        # Evenly spaced indices across teacher layers
        indices = [round(i * (n_t - 1) / (n_s - 1)) for i in range(n_s)]
        copied = skipped = 0
        for s_i, t_i in enumerate(indices):
            s_layer = s_layers[s_i]
            t_layer = t_layers[t_i]
            s_sd = s_layer.state_dict()
            t_sd = t_layer.state_dict()
            compat = {k: v for k, v in t_sd.items()
                      if k in s_sd and v.shape == s_sd[k].shape}
            skipped += len(s_sd) - len(compat)
            s_layer.load_state_dict(compat, strict=False)
            copied += len(compat)
        print(f"  {part:7s}: layers {indices} | {copied} tensors copied | {skipped} skipped")

init_student_from_teacher(teacher_whisper, student_whisper)

t_params = sum(p.numel() for p in teacher_whisper.parameters()) / 1e6
s_params = sum(p.numel() for p in student_whisper.parameters()) / 1e6
vram     = torch.cuda.memory_allocated() / 1e9

print(f"\nTeacher : {t_params:.0f}M params")
print(f"Student : {s_params:.0f}M params")
print(f"Ratio   : {t_params/s_params:.1f}× compression")
print(f"VRAM    : {vram:.2f} GB used")
print("Student ready")

preprocessing and data loaders

In [ ]:
import datasets
datasets.config.AUDIOREAD_DURATION_LIMIT = None

# Force soundfile decoder
import soundfile  # just importing it registers it

In [ ]:
from datasets import concatenate_datasets
from torch.utils.data import DataLoader
import torch
import numpy as np

MAX_AUDIO_S = 15
TRAIN_BATCH = 4
EVAL_BATCH  = 8

ALL_LANGS = ['lug', 'ach', 'nyn', 'teo', 'lgg', 'eng', 'swa']

print("Merging splits across all 7 languages...")
train_ds = concatenate_datasets([salt_splits[l]["train"] for l in ALL_LANGS])
dev_ds   = concatenate_datasets([salt_splits[l]["dev"]   for l in ALL_LANGS])
test_ds  = concatenate_datasets([salt_splits[l]["test"]  for l in ALL_LANGS])

print(f"  Train : {len(train_ds):,} samples")
print(f"  Dev   : {len(dev_ds):,} samples")
print(f"  Test  : {len(test_ds):,} samples")

test_by_lang = {l: salt_splits[l]["test"] for l in ALL_LANGS}

def preprocess_whisper(batch):
    raw = batch["audio"]["array"]
    sr  = batch["audio"]["sampling_rate"]

    if raw is None or len(raw) == 0:
        batch["input_features"] = None
        batch["labels"]         = None
        return batch

    audio = np.array(raw, dtype=np.float32)[:MAX_AUDIO_S * sr]
    audio = np.clip(audio, -1.0, 1.0)

    inputs = processor(audio, sampling_rate=sr, return_tensors="pt")
    batch["input_features"] = inputs.input_features[0]

    labels   = processor.tokenizer(batch["text"]).input_ids
    lang     = batch["audio_language"]
    lang_tok = SALT_LANG_TOKENS.get(lang, SALT_LANG_TOKENS["eng"])
    if labels and labels[0] == processor.tokenizer.bos_token_id:
        labels[0] = lang_tok

    batch["labels"] = labels
    return batch

def filter_invalid(batch):
    return batch["input_features"] is not None and batch["labels"] is not None

def get_remove_cols(ds):
    base     = ["audio", "id"]
    optional = ["is_studio", "speaker_id"]
    return base + [c for c in optional if c in ds.column_names]

print("\nPreprocessing train split (num_proc=1 to surface errors)...")
train_proc = train_ds.map(
    preprocess_whisper,
    remove_columns=get_remove_cols(train_ds),
    num_proc=1,
    desc="Train",
)
train_proc = train_proc.filter(filter_invalid)

print("Preprocessing dev split...")
dev_proc = dev_ds.map(
    preprocess_whisper,
    remove_columns=get_remove_cols(dev_ds),
    num_proc=1,
    desc="Dev",
)
dev_proc = dev_proc.filter(filter_invalid)

train_proc.set_format(type="torch", columns=["input_features"])
dev_proc.set_format(type="torch",   columns=["input_features"])

def collate_fn(batch):
    input_features = torch.stack([b["input_features"] for b in batch])
    label_seqs     = [torch.tensor(b["labels"], dtype=torch.long) for b in batch]
    labels         = torch.nn.utils.rnn.pad_sequence(
        label_seqs, batch_first=True, padding_value=-100
    )
    texts = [b["text"] for b in batch]
    return {"input_features": input_features, "labels": labels, "texts": texts}

train_loader = DataLoader(
    train_proc, batch_size=TRAIN_BATCH, shuffle=True,
    collate_fn=collate_fn, num_workers=2,
    pin_memory=True,
)
dev_loader = DataLoader(
    dev_proc, batch_size=EVAL_BATCH, shuffle=False,
    collate_fn=collate_fn, num_workers=2,
    pin_memory=True,
)

print(f"\nTrain batches : {len(train_loader):,}")
print(f"Dev batches   : {len(dev_loader):,}")
print("DataLoaders ready")

 Loss, optimizer, scheduler

In [ ]:
!pip install -q "transformers==4.39.3" --no-deps

In [ ]:
!pip install -q "transformers==4.39.3"

In [ ]:
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

class WhisperDistillationLoss(nn.Module):
    def __init__(self, temperature=2.0, alpha=0.9):
        super().__init__()
        self.T     = temperature
        self.alpha = alpha

    def forward(self, student_logits, teacher_logits, labels):
        B, T, V = student_logits.shape
        s_flat  = student_logits.view(-1, V)
        t_flat  = teacher_logits.view(-1, V)
        l_flat  = labels.view(-1)
        mask    = l_flat != -100

        hard_loss = F.cross_entropy(s_flat[mask], l_flat[mask])

        s_soft    = F.log_softmax(s_flat[mask] / self.T, dim=-1)
        t_soft    = F.softmax(t_flat[mask]     / self.T, dim=-1)
        soft_loss = F.kl_div(
            s_soft, t_soft, reduction='batchmean'
        ) * (self.T ** 2)

        total = (1 - self.alpha) * hard_loss + self.alpha * soft_loss
        return total, hard_loss, soft_loss

NUM_EPOCHS   = 10
WARMUP_STEPS = 500
TOTAL_STEPS  = len(train_loader) * NUM_EPOCHS

distill_loss = WhisperDistillationLoss(temperature=2.0, alpha=0.9)
optimizer    = AdamW(student_whisper.parameters(), lr=5e-5, weight_decay=0.01)
scheduler    = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=TOTAL_STEPS,
)

print(f"Total steps      : {TOTAL_STEPS:,}")
print(f"Steps per epoch  : {len(train_loader):,}")
print(f"Warmup steps     : {WARMUP_STEPS}")
print(f"Estimated time   : ~{(len(train_loader) * NUM_EPOCHS * 1.2) / 60:.0f} min on T4")
print("Ready to train")

Train loop

In [ ]:
# Step 1: Downgrade numpy in-session (no restart needed)
import subprocess
subprocess.run(["pip", "install", "-q", "numpy==1.26.4"], check=True)

# Step 2: Re-patch the format to numpy instead of torch
train_proc.set_format(type="numpy", columns=["input_features"])
dev_proc.set_format(type="numpy",   columns=["input_features"])

# Step 3: Override collate_fn to convert numpy → torch manually
import numpy as np
import torch

def collate_fn(batch):
    input_features = torch.tensor(
        np.stack([b["input_features"] for b in batch]), dtype=torch.float32
    )
    label_seqs = [torch.tensor(b["labels"], dtype=torch.long) for b in batch]
    labels = torch.nn.utils.rnn.pad_sequence(
        label_seqs, batch_first=True, padding_value=-100
    )
    texts = [b["text"] for b in batch]
    return {"input_features": input_features, "labels": labels, "texts": texts}

# Step 4: Recreate DataLoaders with the fixed collate_fn
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_proc, batch_size=TRAIN_BATCH, shuffle=True,
    collate_fn=collate_fn, num_workers=0,  # 0 is safer after in-session pip install
    pin_memory=True,
)
dev_loader = DataLoader(
    dev_proc, batch_size=EVAL_BATCH, shuffle=False,
    collate_fn=collate_fn, num_workers=0,
    pin_memory=True,
)

print(" Loaders patched — safe to run cell 9 now")

In [ ]:
# ── Monkey-patch numpy's array() to fix the copy=False issue in-memory ──
import numpy as np

_original_array = np.array

def _patched_array(obj, copy=False, **kwargs):
    if not copy:
        return np.asarray(obj, **kwargs)
    return _original_array(obj, copy=copy, **kwargs)

np.array = _patched_array
print("numpy.array patched")

# ── Reset formats and recreate loaders ──
train_proc.set_format(type="numpy", columns=["input_features"])
dev_proc.set_format(type="numpy",   columns=["input_features"])

def collate_fn(batch):
    input_features = torch.tensor(
        np.stack([np.asarray(b["input_features"]) for b in batch]),
        dtype=torch.float32
    )
    label_seqs = [torch.tensor(b["labels"], dtype=torch.long) for b in batch]
    labels = torch.nn.utils.rnn.pad_sequence(
        label_seqs, batch_first=True, padding_value=-100
    )
    texts = [b["text"] for b in batch]
    return {"input_features": input_features, "labels": labels, "texts": texts}

from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_proc, batch_size=TRAIN_BATCH, shuffle=True,
    collate_fn=collate_fn, num_workers=0,  # must be 0 for monkey-patch to apply
    pin_memory=True,
)
dev_loader = DataLoader(
    dev_proc, batch_size=EVAL_BATCH, shuffle=False,
    collate_fn=collate_fn, num_workers=0,
    pin_memory=True,
)

print("Loaders ready — run cell 9 now")

In [ ]:
# ── Fix: include all required columns in set_format ──
train_proc.set_format(type="numpy", columns=["input_features", "labels", "text"])
dev_proc.set_format(type="numpy",   columns=["input_features", "labels", "text"])

def collate_fn(batch):
    input_features = torch.tensor(
        np.stack([np.asarray(b["input_features"]) for b in batch]),
        dtype=torch.float32
    )
    label_seqs = [torch.tensor(b["labels"], dtype=torch.long) for b in batch]
    labels = torch.nn.utils.rnn.pad_sequence(
        label_seqs, batch_first=True, padding_value=-100
    )
    texts = [b["text"] for b in batch]
    return {"input_features": input_features, "labels": labels, "texts": texts}

from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_proc, batch_size=TRAIN_BATCH, shuffle=True,
    collate_fn=collate_fn, num_workers=0,
    pin_memory=True,
)
dev_loader = DataLoader(
    dev_proc, batch_size=EVAL_BATCH, shuffle=False,
    collate_fn=collate_fn, num_workers=0,
    pin_memory=True,
)

print("Ready — run cell 9 now")

In [ ]:
!pip install -q evaluate jiwer

In [ ]:
from evaluate import load as load_metric
import time, os, shutil
from torch.amp import autocast, GradScaler

NUM_EPOCHS  = 3
TRAIN_BATCH = 4
EVAL_BATCH  = 8
CKPT_EVERY  = 2000
KEEP_CKPTS  = 2
LOG_EVERY   = 50
GRAD_ACCUM  = 4

# Move teacher to CPU — only pulled to GPU per batch
teacher_whisper.to("cpu")
torch.cuda.empty_cache()
print(f"VRAM after teacher offload: {torch.cuda.memory_allocated()/1e9:.2f} GB")

from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_proc, batch_size=TRAIN_BATCH, shuffle=True,
    collate_fn=collate_fn, num_workers=0,
    pin_memory=True,
)
dev_loader = DataLoader(
    dev_proc, batch_size=EVAL_BATCH, shuffle=False,
    collate_fn=collate_fn, num_workers=0,
    pin_memory=True,
)

TOTAL_STEPS = NUM_EPOCHS * (len(train_loader) // GRAD_ACCUM)
scaler      = GradScaler("cuda")
wer_metric  = load_metric("wer")
best_wer    = float("inf")
global_step = 0
saved_ckpts = []

print("=" * 60)
print(f"Distillation training — {NUM_EPOCHS} epochs, {TOTAL_STEPS:,} steps")
print(f"7 languages: {', '.join(ALL_LANGS).upper()}")
print(f"Batch size : {TRAIN_BATCH} (accum={GRAD_ACCUM}, effective={TRAIN_BATCH*GRAD_ACCUM})")
print(f"Teacher    : CPU offloaded (pulled to GPU per batch)")
print(f"Mixed precision: ON")
print("=" * 60)

for epoch in range(1, NUM_EPOCHS + 1):
    student_whisper.train()
    epoch_loss = epoch_hard = epoch_soft = 0.0
    t0 = time.time()
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader, 1):
        feats  = batch["input_features"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        # Pull teacher to GPU, run, immediately push back to CPU
        teacher_whisper.to(DEVICE)
        with torch.no_grad():
            with autocast("cuda"):
                t_out = teacher_whisper(
                    input_features=feats.half(), labels=labels
                )
            teacher_logits = t_out.logits.float().detach()
        teacher_whisper.to("cpu")
        torch.cuda.empty_cache()

        # Student forward
        with autocast("cuda"):
            s_out          = student_whisper(input_features=feats, labels=labels)
            student_logits = s_out.logits.float()
            loss, hard, soft = distill_loss(student_logits, teacher_logits, labels)
            loss = loss / GRAD_ACCUM

        scaler.scale(loss).backward()

        if step % GRAD_ACCUM == 0:
            global_step += 1
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(student_whisper.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

            epoch_loss += loss.item() * GRAD_ACCUM
            epoch_hard += hard.item()
            epoch_soft += soft.item()

            if global_step % LOG_EVERY == 0:
                avg_l   = epoch_loss / global_step
                avg_h   = epoch_hard / global_step
                avg_s   = epoch_soft / global_step
                lr      = scheduler.get_last_lr()[0]
                elapsed = (time.time() - t0) / 60
                pct     = step / len(train_loader) * 100
                print(f"  Ep{epoch} [{step:>4}/{len(train_loader)} {pct:4.1f}%] "
                      f"loss={avg_l:.4f} hard={avg_h:.4f} soft={avg_s:.4f} "
                      f"lr={lr:.1e} t={elapsed:.1f}m")

            if global_step % CKPT_EVERY == 0:
                ckpt = f"{OUTPUT_WHISPER}/checkpoint-{global_step}"
                student_whisper.save_pretrained(ckpt)
                saved_ckpts.append(ckpt)
                print(f"  Checkpoint saved → {ckpt}")
                if len(saved_ckpts) > KEEP_CKPTS:
                    old = saved_ckpts.pop(0)
                    if os.path.exists(old):
                        shutil.rmtree(old)
                        print(f"  Deleted old checkpoint → {old}")

    # Epoch eval — teacher stays on CPU during eval
    student_whisper.eval()
    preds_text, refs_text = [], []
    with torch.no_grad():
        for batch in dev_loader:
            with autocast("cuda"):
                ids = student_whisper.generate(
                    batch["input_features"].to(DEVICE),
                    max_new_tokens=225,
                )
            preds_text.extend(
                processor.batch_decode(ids, skip_special_tokens=True)
            )
            refs_text.extend(batch["texts"])

    val_wer  = wer_metric.compute(predictions=preds_text, references=refs_text)
    avg_loss = epoch_loss / max(global_step, 1)
    elapsed  = (time.time() - t0) / 60

    print(f"\n{'='*60}")
    print(f"Epoch {epoch:02d} | loss={avg_loss:.4f} | "
          f"WER={val_wer:.4f} ({val_wer*100:.1f}%) | {elapsed:.1f} min")
    if val_wer < best_wer:
        best_wer = val_wer
        student_whisper.save_pretrained(f"{OUTPUT_WHISPER}/best_model")
        processor.save_pretrained(f"{OUTPUT_WHISPER}/best_model")
        print(f"  Best saved → {OUTPUT_WHISPER}/best_model  (WER={val_wer:.4f})")
    print(f"{'='*60}\n")

print(f"Training complete — Best WER: {best_wer:.4f} ({best_wer*100:.1f}%)")

In [ ]:
print("Evaluating per-language WER on test set...\n")
student_whisper.eval()
results = {}

for lang in ALL_LANGS:
    test_lang = test_by_lang[lang].map(
        preprocess_whisper,
        remove_columns=["audio", "id", "is_studio", "speaker_id"],
        desc=f"{lang} test",
    )
    test_lang.set_format(type="torch", columns=["input_features"])
    loader = DataLoader(test_lang, batch_size=8, collate_fn=collate_fn)

    preds, refs = [], []
    with torch.no_grad():
        for batch in loader:
            ids = student_whisper.generate(
                batch["input_features"].to(DEVICE),
                max_new_tokens=225,
            )
            preds.extend(processor.batch_decode(ids, skip_special_tokens=True))
            refs.extend(test_lang["text"])

    wer            = wer_metric.compute(predictions=preds, references=refs)
    results[lang]  = round(wer * 100, 2)
    filled = int((1 - min(wer, 1.0)) * 25)
    bar    = "█" * filled + "░" * (25 - filled)
    print(f"  {lang.upper():4s}  [{bar}]  WER: {results[lang]:5.1f}%")

avg_wer = sum(results.values()) / len(results)
print(f"\n  Average WER across 7 languages: {avg_wer:.1f}%")
print(f"\n  Summary: {results}")